In [ ]:
#| default_exp handlers.pipeline.loader

# Loader

Compatibility facade for `HandlerConfig`, `load_data`, and `gap_check` during the incremental pipeline modularization.

In [ ]:
#| export
from __future__ import annotations
from marisco.handlers.pipeline.contracts import (
    BasePluginSpec,
    FilePluginSpec,
    HandlerConfig,
    LegacyPathPluginSpec,
    MeltEntry,
    NamePluginSpec,
    PluginSpec,
    PluginSpecModel,
    UnitConversionCfg,
    _MARIS_REQUIRED,
    _MELT_PROVIDES,
)
from marisco.handlers.pipeline.gates import _GapDiagnostics, gap_check
from marisco.handlers.pipeline.intake import IntakePlan, load_data


## Loader Facade

In [ ]:
#| export
from marisco.handlers.pipeline.contracts import (
    BasePluginSpec,
    FilePluginSpec,
    HandlerConfig,
    LegacyPathPluginSpec,
    MeltEntry,
    NamePluginSpec,
    PluginSpec,
    PluginSpecModel,
    UnitConversionCfg,
    _MARIS_REQUIRED,
    _MELT_PROVIDES,
)
from marisco.handlers.pipeline.gates import _GapDiagnostics, gap_check
from marisco.handlers.pipeline.intake import IntakePlan, load_data


In [ ]:
cfg = HandlerConfig.from_yaml("config/handlers/fram_strait.yaml")
print(f"module_name:       {cfg.module_name}")
print(f"url:               {cfg.url[:55]}...")
print(f"rename keys:       {list(cfg.rename)[:3]}")
print(f"melt_spec entries: {len(cfg.melt_spec)}  (first: {cfg.melt_spec[0].val!r})")
print(f"unit_conversions:  {len(cfg.unit_conversions)}  (factor: {cfg.unit_conversions[0].factor:.3e})")
print(f"nuclide_lut:       {cfg.nuclide_lut}")
print(f"col_date / fmt:    {cfg.col_date!r} / {cfg.dt_format!r}")
print("HandlerConfig.from_yaml ✓")

In [ ]:
# Loader path is the only executable plugin surface allowed in YAML
cfg = HandlerConfig.from_yaml("config/handlers/fram_strait.yaml")

# Legacy path: still works unchanged
spec_path = PluginSpec(path="marisco.callbacks.shared.SoftShiftLonCB", args={"shift": 180.0})
assert spec_path.path == "marisco.callbacks.shared.SoftShiftLonCB"
assert spec_path.args == {"shift": 180.0}

# New name: shorthand resolution
spec_name = PluginSpec(name="SoftRegexTransformCB")
assert spec_name.name == "SoftRegexTransformCB"
assert spec_name.path is None

# New file+class: local dynamic load
spec_file = PluginSpec(**{"file": "local_cbs.py", "class": "MyLocalCB"})
assert spec_file.file  == "local_cbs.py"
assert spec_file.class_ == "MyLocalCB"

# Validation guard: bare PluginSpec with no strategy raises
from pydantic import ValidationError
try:
    PluginSpec(args={"x": 1})
    assert False, "Should have raised"
except ValidationError:
    pass

print("PluginSpec ✓ — path/name/file+class all validate; bare spec raises ValidationError")

## load_data

Lazy intake — fetches the provider CSV/TSV and wraps it in the `{grp: DataFrame}` contract.

In [ ]:
#| export
from marisco.handlers.pipeline.intake import (
    IntakePlan,
    _unsupported_default_loader_message,
    call_loader,
    execute_intake_plan,
    load_data,
    resolve_intake_plan,
    resolve_loader_fn,
)


## gap_check

Fail-Fast sensor: raises `ValueError` with scaffold CBs when required MARIS columns will be absent.

In [ ]:
#| export
from marisco.handlers.pipeline.gates import (
    _DEEP_CRITICAL,
    _DEFAULT_LOADER_PATHS,
    _custom_loader_skeleton,
    _deep_gap_message,
    _gate2_skeleton,
    _loader_hint_name,
    _loader_read_hint,
    _loader_suggestion_prefix,
    _stdout_supports,
    _uses_default_loader,
)


In [ ]:
# Valid config: all 7 MARIS columns covered by rename + melt + parse_datetime
cfg_ok = HandlerConfig.from_yaml("config/handlers/fram_strait.yaml")
gap_check(cfg_ok)
print("gap_check(fram_strait) -> passed")


In [ ]:
# Minimal config with no columns at all: fires with skeleton CBs + ValueError
cfg_bad = HandlerConfig(
    module_name="test.handler",
    title="Minimal Test",
    url="http://example.com/data.csv",
    fname_out="test.nc",
)
try:
    gap_check(cfg_bad)
except ValueError as e:
    print(f"ValueError raised: {e}")
